In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs("/content/drive/MyDrive/model_cache", exist_ok=True)
print("✅ Google Drive mounted. Model cache at /content/drive/MyDrive/model_cache")

Mounted at /content/drive
✅ Google Drive mounted. Model cache at /content/drive/MyDrive/model_cache


In [2]:
%pwd

'/content'

In [3]:
import os
from getpass import getpass
os.environ["GEMINI_API_KEY"] = getpass("GEMINI_API_KEY: ")
os.environ["HF_TOKEN"]       = getpass("HF_TOKEN: ")
os.environ["OUTPUT_DIR"]     = "/content/output"
os.makedirs("/content/output", exist_ok=True)

print("✅ Secrets loaded.")
print(f"  GEMINI_API_KEY: {'set ✓' if os.environ.get('GEMINI_API_KEY') else 'MISSING ❌'}")
print(f"  HF_TOKEN:       {'set ✓' if os.environ.get('HF_TOKEN') else 'MISSING ❌'}")

✅ Secrets loaded.
  GEMINI_API_KEY: set ✓
  HF_TOKEN:       set ✓


In [4]:
get_ipython().system("pip install -q transformers diffusers accelerate safetensors huggingface_hub")

# Gemini SDK (NEW — google-generativeai is deprecated)
get_ipython().system("pip install -q google-genai python-dotenv pydantic rich")

# SAM2 (from PyPI)
get_ipython().system("pip install -q sam2")

# Mesh processing
get_ipython().system('pip install -q "trimesh[easy]" pymeshfix pymeshlab open3d xatlas')

# Image processing
get_ipython().system("pip install -q opencv-python-headless scikit-image scipy imageio matplotlib")



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.8/152.8 kB 14.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.5/106.5 MB 8.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 129.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 86.3 MB/s eta 0:00:00
 

In [5]:
# Verify
import torch, transformers, diffusers, sam2, trimesh
print(f"✅ All dependencies installed.")
print(f"   torch={torch.__version__}  CUDA={'✓' if torch.cuda.is_available() else '✗ (CPU)'}")
get_ipython().system("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo 'No GPU'")

✅ All dependencies installed.
   torch=2.10.0+cu128  CUDA=✓
Tesla T4, 15360 MiB


In [7]:
import os, sys

# --- Clone SynergyAmodal ---
if not os.path.exists("/content/SynergyAmodal"):
    get_ipython().system("git clone https://github.com/imlixinyang/SynergyAmodal.git /content/SynergyAmodal")
    print("✅ SynergyAmodal cloned.")
else:
    print("✅ SynergyAmodal already present — skipping clone.")

# --- Install SynergyAmodal requirements ---
# requirements.txt uses modern compatible versions — install as-is.
# Only skip gradio (not needed in pipeline) to save install time.
get_ipython().system(
    "pip install -q "
    "accelerate==1.0.1 "
    "addict==2.4.0 "
    "diffusers==0.31.0 "
    "easydict==1.13 "
    "numpy==1.26.4 "
    "omegaconf==2.3.0 "
    "onnxruntime==1.22.0 "
    "pycocotools==2.0.8 "
    "pytorch_lightning==2.5.0 "
    "transformers==4.45.0"
)

# --- Add to sys.path so imports work at runtime ---
if "/content/SynergyAmodal" not in sys.path:
    sys.path.insert(0, "/content/SynergyAmodal")

# --- Verify core imports ---
try:
    from omegaconf import OmegaConf
    from addict import Dict
    print("✅ SynergyAmodal deps importable — Cell 4 complete.")
except ImportError as e:
    print(f"⚠️  Import check failed: {e}")

✅ SynergyAmodal already present — skipping clone.
✅ SynergyAmodal deps importable — Cell 4 complete.


In [9]:
import os

if not os.path.exists("/content/Hunyuan3D-2"):
    # Step 1: Login to HuggingFace (required for gated Hunyuan3D-2mv model)
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"])

    # Step 2: Clone repo
    get_ipython().system("git clone https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")

    # Step 3: Install requirements
    get_ipython().system("pip install -q -r /content/Hunyuan3D-2/requirements.txt")

    # Step 4: Compile custom CUDA rasterizer (REQUIRED for texture generation)
    try:
        get_ipython().system("cd /content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer && python3 setup.py install")
        get_ipython().system("cd /content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer && bash compile_mesh_painter.sh")
        print("✅ Hunyuan3D CUDA rasterizer compiled successfully.")
    except Exception as e:
        print(f"⚠️  CUDA rasterizer compilation failed: {e}")
        print("   Pipeline will use TRELLIS fallback for 3D reconstruction.")
else:
    print("✅ Hunyuan3D-2 already cloned.")

✅ Hunyuan3D-2 already cloned.


In [ ]:

import os, shutil
from pathlib import Path

os.makedirs("/content/models/sam2", exist_ok=True)
os.makedirs("/content/models/synergyamodal", exist_ok=True)
os.makedirs("/content/models/hunyuan3d", exist_ok=True)

GDRIVE_CACHE = "/content/drive/MyDrive/model_cache"

# --- SAM2 Hiera-Large (~224 MB) ---
sam2_ckpt = "/content/models/sam2/sam2.1_hiera_large.pt"
sam2_gdrive = f"{GDRIVE_CACHE}/sam2/sam2.1_hiera_large.pt"
if not os.path.exists(sam2_ckpt):
    if os.path.exists(sam2_gdrive):
        print("📂 Copying SAM2 from Google Drive cache...")
        os.makedirs("/content/models/sam2", exist_ok=True)
        shutil.copy(sam2_gdrive, sam2_ckpt)
    else:
        print("⬇️  Downloading SAM2 Hiera-Large (~224 MB)...")
        get_ipython().system(f"wget -q -O {sam2_ckpt} https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt")
        os.makedirs(f"{GDRIVE_CACHE}/sam2", exist_ok=True)
        shutil.copy(sam2_ckpt, sam2_gdrive)
print(f"✅ SAM2: {sam2_ckpt}")

# --- SynergyAmodal weights (ldm.ckpt=15.6 GB, vae.ckpt=929 MB, ~16.5 GB total) ---
# Source: https://huggingface.co/cloudyfall/DeoccAnything  (per SynergyAmodal README)
sa_ckpt_dir = "/content/models/synergyamodal"
sa_gdrive   = f"{GDRIVE_CACHE}/synergyamodal"
os.makedirs(sa_ckpt_dir, exist_ok=True)

ldm_ckpt = f"{sa_ckpt_dir}/ldm.ckpt"
vae_ckpt = f"{sa_ckpt_dir}/vae.ckpt"

if not os.path.exists(ldm_ckpt) or not os.path.exists(vae_ckpt):
    if os.path.exists(sa_gdrive) and os.listdir(sa_gdrive):
        print("📂 Copying SynergyAmodal weights from Google Drive cache...")
        import shutil; shutil.copytree(sa_gdrive, sa_ckpt_dir, dirs_exist_ok=True)
    else:
        print("⬇️  Downloading SynergyAmodal ldm.ckpt (~15.6 GB) — this takes ~10-15 min...")
        get_ipython().system(
            f"wget -q --show-progress "
            f"'https://huggingface.co/cloudyfall/DeoccAnything/resolve/main/ldm_ckpt_dir/epoch%3D8-step%3D58000.ckpt?download=true' "
            f"-O {ldm_ckpt}"
        )
        print("⬇️  Downloading SynergyAmodal vae.ckpt (~929 MB)...")
        get_ipython().system(
            f"wget -q --show-progress "
            f"'https://huggingface.co/cloudyfall/DeoccAnything/resolve/main/vae_ckpt_dir/epoch%3D5-step%3D100000.ckpt?download=true' "
            f"-O {vae_ckpt}"
        )
        os.makedirs(sa_gdrive, exist_ok=True)
        import shutil; shutil.copytree(sa_ckpt_dir, sa_gdrive, dirs_exist_ok=True)
        print("💾 Saved SynergyAmodal weights to Google Drive cache.")
print(f"✅ SynergyAmodal: {ldm_ckpt} ({os.path.getsize(ldm_ckpt)/1e9:.1f} GB), {vae_ckpt} ({os.path.getsize(vae_ckpt)/1e6:.0f} MB)")

# --- Hunyuan3D weights (shape + texture) ---
# Shape:   tencent/Hunyuan3D-2mv  → subfolder hunyuan3d-dit-v2-mv       (~10 GB, standard)
# Texture: tencent/Hunyuan3D-2    → subfolder hunyuan3d-paint-v2-0-turbo  (~6 GB)
#
# WHY STANDARD (not Turbo)?
#   Standard uses full 50-step diffusion = better geometric detail on complex
#   surfaces like the camera island/bump. Turbo (4-8 steps) is faster but trades
#   some detail. For product photography fidelity, standard is the right call.
#   To switch to Turbo: change allow_patterns below + subfolder in reconstruction.py.
#
# WHY allow_patterns?
#   snapshot_download with no filter grabs ALL variants in the repo = ~29 GB.
#   allow_patterns restricts it to only the subfolder reconstruction.py actually uses.
#
# WHY cache_dir (not local_dir)?
#   from_pretrained() in reconstruction.py uses cache_dir to find weights.
#   snapshot_download must use the same cache_dir so files are found without re-download.

hunyuan_cache = "/content/models/hunyuan3d"
hunyuan_gdrive = f"{GDRIVE_CACHE}/hunyuan3d"
os.makedirs(hunyuan_cache, exist_ok=True)

# Check if already cached (HF creates model--tencent dirs inside cache_dir)
import glob
shape_cached = bool(glob.glob(f"{hunyuan_cache}/models--tencent--Hunyuan3D-2mv/**", recursive=True))
tex_cached   = bool(glob.glob(f"{hunyuan_cache}/models--tencent--Hunyuan3D-2/**", recursive=True))

if shape_cached and tex_cached:
    print(f"✅ Hunyuan3D weights already in cache: {hunyuan_cache}")
elif os.path.exists(hunyuan_gdrive) and os.listdir(hunyuan_gdrive):
    print("📂 Copying Hunyuan3D from Google Drive cache...")
    shutil.copytree(hunyuan_gdrive, hunyuan_cache, dirs_exist_ok=True)
    print(f"✅ Hunyuan3D: copied from Drive cache")
else:
    from huggingface_hub import snapshot_download

    if not shape_cached:
        print("⬇️  Downloading Hunyuan3D shape (hunyuan3d-dit-v2-mv standard, ~10 GB)...")
        snapshot_download(
            repo_id="tencent/Hunyuan3D-2mv",
            allow_patterns=["hunyuan3d-dit-v2-mv/**"],
            cache_dir=hunyuan_cache,
            token=os.environ["HF_TOKEN"],
        )
        print("✅ Shape downloaded.")

    if not tex_cached:
        print("⬇️  Downloading Hunyuan3D texture (hunyuan3d-paint-v2-0-turbo, ~6 GB)...")
        snapshot_download(
            repo_id="tencent/Hunyuan3D-2",
            allow_patterns=["hunyuan3d-paint-v2-0-turbo/**"],
            cache_dir=hunyuan_cache,
            token=os.environ["HF_TOKEN"],
        )
        print("✅ Texture downloaded.")

    os.makedirs(hunyuan_gdrive, exist_ok=True)
    shutil.copytree(hunyuan_cache, hunyuan_gdrive, dirs_exist_ok=True)
    print("💾 Saved Hunyuan3D to Google Drive cache.")

print(f"✅ Hunyuan3D cache: {hunyuan_cache}")


# change the in painting model to tencent/Hunyuan3D-2.1/hunyuan3d-paintpbr-v2-1 -->only 6 gbs rather than 14
# thedisk size in the google collab system gets filled so we have to dothis next time/


✅ SAM2: /content/models/sam2/sam2.1_hiera_large.pt
✅ SynergyAmodal: /content/models/synergyamodal/ldm.ckpt (15.6 GB), /content/models/synergyamodal/vae.ckpt (929 MB)
✅ Hunyuan3D weights already in cache: /content/models/hunyuan3d
✅ Hunyuan3D cache: /content/models/hunyuan3d


In [15]:
import sys, os

REPO_URL = "https://github.com/YOUR_USERNAME/Image-to-3d-model.git"  # ← update this
REPO_DIR = "/content/Image-to-3d-model"

if not os.path.exists(REPO_DIR):
    get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    print(f"✅ Cloned repo to {REPO_DIR}")
else:
    get_ipython().system(f"cd {REPO_DIR} && git pull")
    print(f"✅ Repo up-to-date at {REPO_DIR}")

# Add repo to Python path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Set environment overrides for Colab paths
os.environ["SAM2_CHECKPOINT_DIR"]    = "/content/models/sam2"
os.environ["SYNERGYAMODAL_CKPT_DIR"] = "/content/models/synergyamodal"
os.environ["HUNYUAN3D_CACHE_DIR"]    = "/content/models/hunyuan3d"
os.environ["OUTPUT_DIR"]             = "/content/output"
os.environ["SAM2_VARIANT"]           = "large"

print("✅ Environment configured.")

Cloning into '/content/Image-to-3d-model'...
fatal: could not read Username for 'https://github.com': No such device or address
✅ Cloned repo to /content/Image-to-3d-model
✅ Environment configured.


In [16]:
# save this notebook